In [1]:
import os
import json
import random
import urllib.request
import zipfile
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path, PurePosixPath
 
import torch, torchvision
torchvision.disable_beta_transforms_warning()
import numpy as np
import pandas as pd
import h5py
from PIL import Image
from tqdm.auto import tqdm
from google.cloud import storage
from transformers import AutoImageProcessor, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

In [5]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
DATASET = 'SnapshotSerengetiS02'
METADATA_URL = "https://lilawildlife.blob.core.windows.net/lila-wildlife/snapshotserengeti-v-2-0/SnapshotSerengetiS02.json.zip"
ZIP_FILE = "snapshot_serengeti_s2.json.zip"
JSON_FILE = f"./json_files/{DATASET}.json"
MD_JSON = f"./json_files/SnapshotSerengeti_md.json"
DATA_DIR = "../../../media/Data-10T-1/Bhavesh-project/ser_data"
OUTPUT_H5 = f"../embeddings/{DATASET}_behavior_single_embeddings.h5"
 
SAMPLES_PER_BEHAVIOR = 1000
BATCH_SIZE = 32
NUM_WORKERS = 8
PADDING_FRACTION = 0.10
pos_threshold = 0.5
DINOV2_MODEL = "facebook/dinov2-large"
ANIMAL_CATEGORY = "1"
EMPTY_LABELS = {"empty", "human", "blank"}
BEHAVIOR_COLS = ["standing", "resting", "moving", "interacting", "young_present"]
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(DATA_DIR, exist_ok=True)

for behavior in BEHAVIOR_COLS:
    os.makedirs(os.path.join(DATA_DIR, behavior), exist_ok=True)

In [3]:
# ─── CROPPING HELPERS ──────────────────────────────────────────────────────────
def dataset_path_to_local_name(path_like: str) -> str:
    normalized = str(path_like).replace("\\", "/")
    return str(PurePosixPath(normalized)).replace("/", "_")
 
def choose_square_start(center: float, side: int, limit: int) -> int:
    if side >= limit: return 0
    return min(max(int(round(center - side / 2)), 0), limit - side)
 
def crop_with_padding(image: Image.Image, bbox: list, pad: float = PADDING_FRACTION) -> Image.Image:
    bbox = [float(v) for v in bbox]
    img_w, img_h = image.size
    x_min = min(max(bbox[0], 0.0), 1.0) * img_w
    y_min = min(max(bbox[1], 0.0), 1.0) * img_h
    box_w, box_h = bbox[2] * img_w, bbox[3] * img_h
 
    pad_x, pad_y = pad * box_w, pad * box_h
    side = min(max(box_w + 2 * pad_x, box_h + 2 * pad_y), float(min(img_w, img_h)))
    side_px = max(1, min(int(round(side)), img_w, img_h))
 
    x0 = choose_square_start(x_min + box_w / 2, side_px, img_w)
    y0 = choose_square_start(y_min + box_h / 2, side_px, img_h)
    return image.crop((x0, y0, x0 + side_px, y0 + side_px))

In [6]:
# ─── STEP 1: LOAD MEGADETECTOR ───────────────────────────────────────────────
print("Loading MegaDetector Map...")
with open(MD_JSON) as f:
    md = json.load(f)

md_lookup = {}
for image in md["images"]:
    detections = image.get("detections") or []  # handles missing, null, and empty
    animal_dets = [d for d in detections if str(d.get("category")) == ANIMAL_CATEGORY and "bbox" in d]
    if animal_dets:
        local_name = dataset_path_to_local_name(image["file"])
        md_lookup[local_name] = max(animal_dets, key=lambda d: float(d["conf"]))
        
print('Done')
print(len(md_lookup))

In [7]:
# ─── STEP 2: BEHAVIOR SAMPLING (DYNAMICALLY BALANCED 1000 TOTAL) ───────────────
if not os.path.exists(JSON_FILE):
    print("Downloading metadata...")
    urllib.request.urlretrieve(METADATA_URL, ZIP_FILE)
    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        z.extractall(os.path.dirname(JSON_FILE))
    os.remove(ZIP_FILE)

print("Loading Dataset Metadata...")
with open(JSON_FILE) as f:
    metadata = json.load(f)

cat_to_name = {c["id"]: c["name"].lower().strip() for c in metadata["categories"]}
image_map = {img["id"]: img for img in metadata["images"]}

valid_pool = []
for ann in metadata["annotations"]:
    species = cat_to_name.get(ann["category_id"], "unknown")
    if species in EMPTY_LABELS or species == "unknown": continue
    if any(pd.isna(ann.get(col)) for col in BEHAVIOR_COLS): continue

    img = image_map.get(ann["image_id"])
    if not img: continue

    lname = dataset_path_to_local_name(img["file_name"])
    if lname in md_lookup:
        valid_pool.append((ann, img, lname, species))
        
print(len(valid_pool))

In [8]:
random.seed(42)
download_registry = {}

for target_behavior in BEHAVIOR_COLS:
    behavior_dir = os.path.join(DATA_DIR, target_behavior)
    os.makedirs(behavior_dir, exist_ok=True)
    
    # Separate Positive and Negative groups
    pos_candidates = [x for x in valid_pool if float(x[0].get(target_behavior, 0.0)) > pos_threshold]
    neg_candidates = [x for x in valid_pool if float(x[0].get(target_behavior, 0.0)) <= pos_threshold]
    
    # Calculate balance bounds dynamically
    target_pos = SAMPLES_PER_BEHAVIOR // 2
    target_neg = SAMPLES_PER_BEHAVIOR // 2
    if len(pos_candidates) < target_pos:
        target_pos = len(pos_candidates)
        target_neg = min(len(neg_candidates), SAMPLES_PER_BEHAVIOR - target_pos)
    elif len(neg_candidates) < target_neg:
        target_neg = len(neg_candidates)
        target_pos = min(len(pos_candidates), SAMPLES_PER_BEHAVIOR - target_neg)
        
    sampled_pos = random.sample(pos_candidates, target_pos)
    sampled_neg = random.sample(neg_candidates, target_neg)
    
    # Map allocations
    behavior_records = {}
    for ann, img, lname, spec in (sampled_pos + sampled_neg):
        is_active = 1 if float(ann.get(target_behavior, 0.0)) > pos_threshold else 0
        behavior_records[lname] = {
            "file_name": img["file_name"],
            "species": spec,
            "label": is_active
        }
    download_registry[target_behavior] = behavior_records

# Execute Parallel Downloads
print("\nScanning folders for download queues...")
bucket = storage.Client.create_anonymous_client().bucket("public-datasets-lila")

def dl(t):
    try: bucket.blob(t[0]).download_to_filename(t[1]); return True
    except: return False

for target_behavior in BEHAVIOR_COLS:
    behavior_dir = os.path.join(DATA_DIR, target_behavior)
    download_tasks = []
    
    for lname, data in download_registry[target_behavior].items():
        local_path = os.path.join(behavior_dir, lname)
        if not os.path.exists(local_path):
            download_tasks.append(("snapshotserengeti-unzipped/" + data["file_name"], local_path))
            
    if download_tasks:
        print(f"Downloading {len(download_tasks)} frames for folder '{target_behavior}'...")
        with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
            list(tqdm(pool.map(dl, download_tasks), total=len(download_tasks), desc=f"DL {target_behavior}"))
    else:
        print(f"-> All {target_behavior} images are already on disk.")

In [9]:
print("Initializing DINOv2 Architecture...")
processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL)
model = AutoModel.from_pretrained(DINOV2_MODEL).to(DEVICE).eval()

for target_behavior in BEHAVIOR_COLS:
    behavior_dir = os.path.join(DATA_DIR, target_behavior)
    output_h5_file = f"../embeddings/{DATASET}_{target_behavior}_embeddings.h5"
    os.makedirs(os.path.dirname(output_h5_file), exist_ok=True)
    
    records = download_registry[target_behavior]
    work = [(Path(behavior_dir) / ln, records[ln], md_lookup[ln]) for ln in records if (Path(behavior_dir) / ln).exists()]
    
    if not work:
        print(f"Missing image directory for {target_behavior}. Skipping embedding step.")
        continue
        
    print(f"\nProcessing feature extraction for '{target_behavior}' ({len(work)} elements)...")
    
    with h5py.File(output_h5_file, "w") as hf:
        hf.create_dataset("embeddings", (0, 1024), maxshape=(None, 1024), dtype="float32", chunks=(64, 1024))
        hf.create_dataset("label", (0,), maxshape=(None,), dtype="int32", chunks=(64,))
        hf.create_dataset("species", (0,), maxshape=(None,), dtype=h5py.string_dtype(), chunks=(64,))
        
        def flush(crops, meta):
            inputs = processor(images=crops, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                emb = model(**inputs).last_hidden_state[:, 0, :].cpu().float().numpy()
            lbls = np.array([m["label"] for m in meta], dtype=np.int32)
            spp = np.array([m["species"] for m in meta], dtype=h5py.string_dtype())
            
            n = len(hf["embeddings"])
            for k, v in [("embeddings", emb), ("label", lbls), ("species", spp)]:
                hf[k].resize(n + len(crops), axis=0)
                hf[k][n : n + len(crops)] = v
            crops.clear(); meta.clear()

        b_crops, b_meta = [], []
        for lp, rec, dt in tqdm(work, desc=f"Embedding {target_behavior}"):
            try:
                with Image.open(lp) as im:
                    b_crops.append(crop_with_padding(im.convert("RGB"), dt["bbox"]))
                b_meta.append(rec)
            except Exception as e:
                continue
            if len(b_crops) >= BATCH_SIZE:
                flush(b_crops, b_meta)
        if b_crops:
            flush(b_crops, b_meta)
            
print("\nExtraction complete! All isolated HDF5 files are saved.")

In [10]:
import os
import h5py
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

results = []

for target_behavior in BEHAVIOR_COLS:
    output_h5_file = f"../embeddings/{DATASET}_{target_behavior}_embeddings.h5"

    if not os.path.exists(output_h5_file):
        print(f"Skipping evaluation for {target_behavior}: HDF5 file missing.")
        continue

    with h5py.File(output_h5_file, "r") as hf:
        X = hf["embeddings"][:]
        y = hf["label"][:]
        
    pos_count = int((y == 1).sum())
    neg_count = int((y == 0).sum())
    total_count = len(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=42
    )

    models = {
        "kNN": KNeighborsClassifier(n_neighbors=5, weights="distance", n_jobs=-1),
        "LogReg": LogisticRegression(max_iter=1000, random_state=42),
        "SVM": SVC(probability=True, random_state=42),
    }

    row = {
        "Behavior": target_behavior,
        "Total Samples": total_count,
        "Positive": pos_count,
        "Negative": neg_count,
    }

    for model_name, clf in models.items():
        clf.fit(X_train, y_train)

        preds = clf.predict(X_test)
        proba = clf.predict_proba(X_test)[:, 1]

        row[f"{model_name} AUC-ROC"] = roc_auc_score(y_test, proba)

    results.append(row)

comparison_table = pd.DataFrame(results).sort_values("Behavior").reset_index(drop=True)

print("\nModel comparison across behaviors:")
print(comparison_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
########################
#### Adaptive Model ####
########################

In [ ]:
# =========================
# Cell 1: Imports
# =========================
import os
import json
import random
import zipfile
import urllib.request
import numpy as np
import pandas as pd

In [ ]:
# =========================
# Cell 2: Load metadata if needed
# =========================
if not os.path.exists(JSON_FILE):
    print("Downloading metadata...")
    urllib.request.urlretrieve(METADATA_URL, ZIP_FILE)
    with zipfile.ZipFile(ZIP_FILE, "r") as z:
        z.extractall(os.path.dirname(JSON_FILE))
    os.remove(ZIP_FILE)

print("Loading Dataset Metadata...")
with open(JSON_FILE) as f:
    metadata = json.load(f)

cat_to_name = {c["id"]: c["name"].lower().strip() for c in metadata["categories"]}
image_map = {img["id"]: img for img in metadata["images"]}
print('Done')

In [ ]:
# =========================
# Cell 3: Load MegaDetector lookup
# =========================
print("Loading MegaDetector Map...")
with open(MD_JSON) as f:
    md = json.load(f)

md_lookup = {}
for image in md["images"]:
    detections = image.get("detections") or []
    animal_dets = [
        d for d in detections
        if str(d.get("category")) == ANIMAL_CATEGORY and "bbox" in d
    ]
    if animal_dets:
        local_name = dataset_path_to_local_name(image["file"])
        md_lookup[local_name] = max(animal_dets, key=lambda d: float(d["conf"]))

print("Done")
print(f"MegaDetector usable detections: {len(md_lookup)}")

In [ ]:
# =========================
# Cell 4: Build full valid pool once
# =========================
valid_pool = []

for ann in metadata["annotations"]:
    species = cat_to_name.get(ann["category_id"], "unknown")
    if species in EMPTY_LABELS or species == "unknown":
        continue
    if any(pd.isna(ann.get(col)) for col in BEHAVIOR_COLS):
        continue

    img = image_map.get(ann["image_id"])
    if not img:
        continue

    lname = dataset_path_to_local_name(img["file_name"])
    if lname not in md_lookup:
        continue

    valid_pool.append({
        "image_id": ann["image_id"],
        "file_name": img["file_name"],
        "local_name": lname,
        "species": species,
        **{col: float(ann.get(col, 0.0)) for col in BEHAVIOR_COLS}
    })

valid_df = pd.DataFrame(valid_pool)

print(f"Total valid samples after filtering: {len(valid_df)}")
display(valid_df.head())

In [ ]:
# =========================
# Cell 5: Pre-training data analysis
# This is the key analysis you asked for.
# =========================
analysis_rows = []

for behavior in BEHAVIOR_COLS:
    pos_mask = valid_df[behavior] > pos_threshold
    pos_count = int(pos_mask.sum())
    neg_count = int((~pos_mask).sum())
    total = pos_count + neg_count

    analysis_rows.append({
        "Behavior": behavior,
        "Total Available": total,
        "Positive Available": pos_count,
        "Negative Available": neg_count,
        "Positive %": pos_count / total if total else np.nan,
        "Negative %": neg_count / total if total else np.nan,
        "Max Balanced Total": 2 * min(pos_count, neg_count),
        "Enough for 1000?": (min(pos_count, neg_count) >= SAMPLES_PER_BEHAVIOR // 2),
    })

availability_table = pd.DataFrame(analysis_rows).sort_values("Behavior").reset_index(drop=True)

print("Available class counts before any sampling or training:")
display(availability_table)